Some of the content here is adapted from [LangChain Handbook](https://github.com/pinecone-io/examples/tree/master/generation/langchain/handbook)

Make sure you install langchain and openai packages

`pip install openai langchain langchain_openai langchain_community langchain_experimental docarray`

In [1]:
from langchain_openai import ChatOpenAI
import langchain
langchain.__version__

'1.2.10'

A Large Language Model (LLM) is a type of artificial intelligence (AI) model designed to "understand", generate, and manipulate human language. LLMs are trained on vast amounts of text data, enabling them to learn the intricacies of language, including grammar, context, and even the subtleties of different writing styles. These models operate on the principles of machine learning, particularly deep learning, using architectures such as transformers to process and predict text sequences.

At their core, LLMs use a neural network architecture called the transformer. This architecture allows the model to weigh the importance of different words in a sentence, understanding both their meaning in isolation and how they relate to each other. The training process involves feeding the model examples of text and adjusting its parameters (through a process called backpropagation) to minimize the difference between its predictions and the actual outcomes.

Without getting into the technical details, we will focus on the applications of LLMs for data and business analytics problems.

# OpenAI



In [2]:
#os.environ['OPENAI_API_KEY'] = "SETUP YOUR KEY IF NOT ALREADY THERE"
llm = ChatOpenAI(temperature=0, model="gpt-5-mini")

In [3]:
llm.invoke("Tell me a joke about Dr. Safadi who works in the University of Georgia and teaches advanced data and analytics")

AIMessage(content='1) Why did Dr. Safadi bring a ladder to his Advanced Data & Analytics class at UGA?  \nBecause today’s topic was hierarchical clustering — and he wanted the students to get to the top level!\n\n2) Dr. Safadi doesn’t grade on a curve—he fits the curve, explains 95% of the variance, and calls it a confidence interval.\n\n3) A student asked Dr. Safadi if their model would generalize. He replied, “Only if it leaves the training set and makes friends at the tail ends.”\n\n4) At the University of Georgia, when Dr. Safadi says a result is “statistically significant,” even the Bulldogs cheer.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 531, 'prompt_tokens': 28, 'total_tokens': 559, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name

# Intro to LangChain

LangChain is a popular framework that allow users to quickly build apps and pipelines around **L**arge **L**anguage **M**odels. It can be used to for chatbots, **G**enerative **Q**uestion-**A**nwering (GQA), summarization, and much more.

The core idea of the library is that we can _"chain"_ together different components to create more advanced use-cases around LLMs. Chains may consist of multiple components from several modules:

* **Prompt templates**: Prompt templates are, well, templates for different types of prompts. Like "chatbot" style templates, ELI5 question-answering, etc

* **LLMs**: Large language models like GPT-3, BLOOM, etc

* **Agents**: Agents use LLMs to decide what actions should be taken, tools like web search or calculators can be used, and all packaged into logical loop of operations.

* **Memory**: Short-term memory, long-term memory.

# Prompts

In [4]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("tell me a short joke about {topic}")
prompt

ChatPromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='tell me a short joke about {topic}'), additional_kwargs={})])

In [5]:
prompt_value = prompt.invoke({"topic": "ice cream"})
prompt_value

ChatPromptValue(messages=[HumanMessage(content='tell me a short joke about ice cream', additional_kwargs={}, response_metadata={})])

In [6]:
prompt_value.to_string()

'Human: tell me a short joke about ice cream'

# Output Parser

In [7]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tracers import ConsoleCallbackHandler
output_parser = StrOutputParser()
output_parser

StrOutputParser()

In [8]:
output_parser.invoke("Hello")

'Hello'

# Chain

`chain = prompt | model | output_parser`

In [9]:
chain = prompt | llm | output_parser

In [10]:
chain.invoke({"topic": "ice cream"})

'Why did the ice cream cone get promoted? Because it was always on a roll.'

In [11]:
chain.invoke({"topic": "ice cream"},
             config={'callbacks': [ConsoleCallbackHandler()]}
            )

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "topic": "ice cream"
}
[chain/start] [chain:RunnableSequence > prompt:ChatPromptTemplate] Entering Prompt run with input:
{
  "topic": "ice cream"
}
[chain/end] [chain:RunnableSequence > prompt:ChatPromptTemplate] s] Exiting Prompt run with output:
[outputs]
[llm/start] [chain:RunnableSequence > llm:ChatOpenAI] Entering LLM run with input:
{
  "prompts": [
    "Human: tell me a short joke about ice cream"
  ]
}
[llm/end] [chain:RunnableSequence > llm:ChatOpenAI] [6.23s] Exiting LLM run with output:
{
  "generations": [
    [
      {
        "text": "Why did the ice cream cone cross the road? Because it was a sucker.",
        "generation_info": {
          "finish_reason": "stop",
          "logprobs": null
        },
        "type": "ChatGeneration",
        "message": {
          "lc": 1,
          "type": "constructor",
          "id": [
            "langchain",
            "schema",
            "messages",
  

'Why did the ice cream cone cross the road? Because it was a sucker.'

In [12]:
chain.invoke({"topic": "Dr. Safadi who works in the University of Georgia and teaches advanced data and analytics"})

'Why did Dr. Safadi at UGA cross the road? To get a better sample.'

# Applications

## Zero-shot learning

In [13]:
prompt = ChatPromptTemplate.from_template("Summarize the text below into two sentences:\n{text}")
chain = prompt | llm | output_parser

In [14]:
# https://www.cnn.com/2023/12/16/business/shawn-fain-4-day-work-week/index.html
text = """
New York CNN  — 

Of all the bargaining goals set out by the United Auto Workers union in the just completed talks, none was more ambitious, or gained less traction in negotiations, than the idea of a four-day, 32-hour work week.

Without a cut in pay, too.

That didn’t happen, at least not this round. And it might sound like an impossible dream for most American workers. But UAW President Shawn Fain says that it’s not just achievable, but a dream that his union forefathers believed in.

“I think it’s a very realistic goal,” he said in an interview with CNN Business. He points out that it was a negotiating goal of the union back in the middle of the last century, soon after it won the right to represent workers at the nation’s automakers.

“I don’t know what happened over the next 60 or 70 years, but that conversation fell by the wayside,” he said. “So I felt it was imperative that we get the dialogue going back to, you know, workers reclaiming their lives,” he said.

But the idea was dismissed by automakers. Some executives, such as Ford CEO Jim Farley, said agreeing to that would lead to massive losses for the automakers.

“If we had done that [four-day week]….we would have gone bankrupt many years ago,” said Farley to CNN on September 14, the day before the strike started. “We’d have to close plants and most people would lose their jobs.”

And once the union and automakers started making progress toward the deal that would eventually end the strike, there was little discussion of a four-day week ever again. The union did win a record contract, with an immediate wage gain of at least 11%, an additional 14 percentage points of raises throughout the life of the contract, a return of cost-of-living adjustments (COLA) and improved retirement benefits, among other gains.

Most hourly workers at GM, Ford and Stellantis will be paid at least $43 an hour by the time the recently ratified contract ends in April 2028. That comes to about $1,700 a week for a 40-hour week. And the COLA is likely to raise it further than that. And while a hypothetical pay structure for an alternate deal featuring a four-day work week may be unknowable, paying that weekly wage for a 32-hour week would increase the hourly wage by about 25%, on top of the just negotiated wage hikes.

Whether the companies would be able to afford that additional increase will depend on a number of factors, including productivity changes, demand for vehicles and the impact that would have on pricing and the cost of raw materials, which ultimately have a far greater impact on the cost of production than labor costs. Labor accounts for only about 7% to 8% of the cost of a vehicle. """

In [15]:
chain.invoke({"text": text})

'The UAW pushed for a four-day, 32-hour workweek without a pay cut—a historically rooted but ambitious demand that automakers dismissed as financially untenable and that gained no traction in the final talks. Instead the union won a record contract with immediate and scheduled wage increases, COLA and better retirement benefits; while paying the same weekly wage over 32 hours would raise hourly pay by roughly 25%, whether automakers could absorb that depends on productivity, vehicle demand and pricing (labor is only about 7–8% of a vehicle’s cost).'

## One-shot learning

In [16]:
prompt = ChatPromptTemplate.from_template(
    """Copyedit the text delimited correcting any typos and rewriting it in fluent American English.
    [TEXT]: The prices of used car have increase during pandemc because people started to drive not fly
    [COPYEDIT]: The prices of used cars have increased during the pandemic because people have started to drive instead of flying.
    [TEXT]: {text}
    [COPYEDIT]:"""
)
chain = prompt | llm | output_parser


In [17]:
chain.invoke({"text":"Python is a great probraming language especially fordata science"})

'Python is a great programming language, especially for data science.'

## Few-shot learning

In [18]:
prompt = ChatPromptTemplate.from_template(
    """"Classify the following sentences as either 'Positive' or 'Negative' sentiment."
Sentence: "I absolutely love the friendly staff and the warm atmosphere at this café."
Classification: Positive
Sentence: "The movie was a complete waste of my time and money."
Classification: Negative
Sentence: "The book was okay, but not as exciting as I hoped it would be."
Classification: Negative
Sentence: "Her performance in the play was absolutely breathtaking and unforgettable."
Classification: Positive
Sentence: "This new software makes my work so much easier and more efficient."
Classification: Positive
Sentence: "{sentence}"
Classification:""")
chain = prompt | llm | output_parser

In [19]:
tweets = [
    "Just finished a session on Python with Prof. Hani Safadi. #DataScience is challenging but exciting!",
    "Prof. Hani Safadi makes learning Python fun! Really enjoying his data science lectures. #PythonRocks",
    "Data science with Python is not easy, but Prof. Safadi's clear explanations help a lot. #AlwaysLearning",
    "Today's Python class with Prof. Safadi left me more challenged than ever. #DataScienceStruggles"
]

In [20]:
for tweet in tweets:
    print(tweet, chain.invoke({"sentence":tweet}))

Just finished a session on Python with Prof. Hani Safadi. #DataScience is challenging but exciting! Classification: Positive
Prof. Hani Safadi makes learning Python fun! Really enjoying his data science lectures. #PythonRocks Classification: Positive
Data science with Python is not easy, but Prof. Safadi's clear explanations help a lot. #AlwaysLearning Sentence: "Data science with Python is not easy, but Prof. Safadi's clear explanations help a lot. #AlwaysLearning"
Classification: Positive
Today's Python class with Prof. Safadi left me more challenged than ever. #DataScienceStruggles Sentence: "Today's Python class with Prof. Safadi left me more challenged than ever. #DataScienceStruggles"
Classification: Negative
